In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

source = "/Volumes/workspace/default/raw/yellow/"
checkpoint = "/Volumes/workspace/default/raw/_checkpoints/bronze_trips"

q = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", checkpoint)
        .load(source)
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingested_at", F.current_timestamp())
     .writeStream
        .option("checkpointLocation", checkpoint)
        .trigger(availableNow=True)
        .toTable("workspace.bronze.trips"))

q.awaitTermination()

In [0]:
%sql
SELECT year, month, COUNT(*) AS trips
FROM workspace.bronze.trips
GROUP BY year, month
ORDER BY year, month